# Hybrid SCD Pipeline — YOLO Detector + R-CNN Classifier

This notebook connects both trained models into a single end-to-end pipeline:

```
Blood Smear Image
       |
       v
  [YOLOv8 Detector]  -->  Bounding boxes around cells
       |
       v
  [Crop each cell]   -->  Individual cell images
       |
       v
  [R-CNN Classifier]  -->  Normal / Sickle per cell
       |
       v
  [Aggregate Results]  -->  Cell counts, sickle ratio, diagnosis
```

## Models Required
- `scd_yolov8_local_detect.pt` — YOLOv8 detector (from `yolo_local.ipynb`)
- `scd_rcnn_local_classifier.pt` — R-CNN classifier (from `rcnn_local.ipynb`)

## Usage
Point `TEST_IMAGES` to any blood smear image(s) and run all cells.

In [3]:
# ============================================================
# CELL 1: Setup
# ============================================================
!pip -q install ultralytics

import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from torchvision import transforms, models
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

MessageError: User cancelled dfs_ephemeral authorization

In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================

# --- Model paths ---
YOLO_MODEL_PATH = '/content/drive/MyDrive/SCD/scd_yolov8_local_detect.pt'
RCNN_MODEL_PATH = '/content/drive/MyDrive/SCD/scd_rcnn_local_classifier.pt'

# --- Test images (change this to your images) ---
# Can be a folder path or a list of image paths
TEST_IMAGES_DIR = '/content/drive/MyDrive/SCD/SCD_LocalDataset/test/images'

# --- Inference settings ---
YOLO_CONF_THRESH = 0.25      # Minimum detection confidence
YOLO_IOU_THRESH  = 0.45      # NMS IoU threshold
YOLO_IMGSZ       = 640       # YOLO input size
RCNN_INPUT_SIZE  = 224       # R-CNN input size
CROP_PADDING     = 0.1       # 10% padding around detected boxes

# --- Diagnosis threshold ---
SICKLE_RATIO_THRESHOLD = 0.3  # If >30% cells are sickle -> SCD Positive

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Configuration loaded.')
print(f'  YOLO model: {YOLO_MODEL_PATH}')
print(f'  R-CNN model: {RCNN_MODEL_PATH}')
print(f'  Test images: {TEST_IMAGES_DIR}')

In [ ]:
# ============================================================
# CELL 3: Load Both Models
# ============================================================

# --- Load YOLO Detector ---
print('Loading YOLO detector...')
detector = YOLO(YOLO_MODEL_PATH)
yolo_class_names = detector.names
print(f'  YOLO classes: {yolo_class_names}')
print(f'  Task: {detector.task}')

# --- Load R-CNN Classifier ---
print('\nLoading R-CNN classifier...')
classifier = torch.load(RCNN_MODEL_PATH, weights_only=False)
classifier.eval()
classifier = classifier.to(DEVICE)

# Get the number of output classes from the model's last layer
# Walk backwards through classifier head to find the last Linear layer
rcnn_num_classes = None
for module in reversed(list(classifier.classifier.modules())):
    if isinstance(module, nn.Linear):
        rcnn_num_classes = module.out_features
        break

print(f'  R-CNN output classes: {rcnn_num_classes}')
print(f'  Total parameters: {sum(p.numel() for p in classifier.parameters()):,}')

# --- R-CNN class names ---
# These must match what was used during training.
# The R-CNN was trained on crops from the local dataset,
# so its classes come from ImageFolder's alphabetical sorting
# of the crop folder names (which match data.yaml names).
# If your data.yaml has: {0: 'Normal', 1: 'Sickle'}
# then ImageFolder will produce: {'Normal': 0, 'Sickle': 1}
# Adjust this list if your classes are different.
import yaml
data_yaml = os.path.join('/content/drive/MyDrive/SCD/SCD_LocalDataset', 'data.yaml')
if os.path.exists(data_yaml):
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    raw_names = cfg.get('names', [])
    if isinstance(raw_names, dict):
        raw_names = [raw_names[i] for i in sorted(raw_names.keys())]
    # ImageFolder sorts alphabetically
    RCNN_CLASS_NAMES = sorted(raw_names)
else:
    RCNN_CLASS_NAMES = ['Normal', 'Sickle']

print(f'  R-CNN class names (sorted): {RCNN_CLASS_NAMES}')
print('\nBoth models loaded successfully.')

In [ ]:
# ============================================================
# CELL 4: Define Hybrid Pipeline
# ============================================================

# Preprocessing transform for R-CNN input
rcnn_transform = transforms.Compose([
    transforms.Resize((RCNN_INPUT_SIZE, RCNN_INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def run_hybrid_pipeline(image_path, detector, classifier, class_names,
                        conf_thresh=0.25, iou_thresh=0.45, padding=0.1):
    """
    Run the full hybrid pipeline on a single image.

    Args:
        image_path: Path to a blood smear image
        detector: YOLO detection model
        classifier: R-CNN classifier model
        class_names: List of class names for classifier output
        conf_thresh: YOLO confidence threshold
        iou_thresh: YOLO NMS IoU threshold
        padding: Fraction of box size to add as padding

    Returns:
        dict with:
          - 'image': PIL Image
          - 'detections': list of dicts with box, yolo_conf, class, class_conf
          - 'summary': aggregated counts and diagnosis
    """
    # --- Stage 1: YOLO Detection ---
    results = detector.predict(
        image_path,
        imgsz=YOLO_IMGSZ,
        conf=conf_thresh,
        iou=iou_thresh,
        device=YOLO_DEVICE,
        verbose=False
    )
    r = results[0]

    img = Image.open(image_path).convert('RGB')
    w_img, h_img = img.size

    detections = []

    if r.boxes is None or len(r.boxes) == 0:
        return {
            'image': img,
            'detections': [],
            'summary': {
                'total_cells': 0,
                'counts': {},
                'sickle_ratio': 0.0,
                'diagnosis': 'No cells detected'
            }
        }

    # --- Stage 2: Crop & Classify each detection ---
    crop_tensors = []
    box_data = []

    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        yolo_conf = box.conf[0].item()
        yolo_cls = int(box.cls[0].item())

        # Add padding
        bw = x2 - x1
        bh = y2 - y1
        pad_w = bw * padding
        pad_h = bh * padding

        cx1 = max(0, int(x1 - pad_w))
        cy1 = max(0, int(y1 - pad_h))
        cx2 = min(w_img, int(x2 + pad_w))
        cy2 = min(h_img, int(y2 + pad_h))

        # Crop
        crop = img.crop((cx1, cy1, cx2, cy2))

        # Skip tiny crops
        if crop.size[0] < 10 or crop.size[1] < 10:
            continue

        # Preprocess for R-CNN
        crop_tensor = rcnn_transform(crop)
        crop_tensors.append(crop_tensor)
        box_data.append({
            'box': [float(x1), float(y1), float(x2), float(y2)],
            'yolo_conf': yolo_conf,
            'yolo_cls': yolo_cls,
        })

    if not crop_tensors:
        return {
            'image': img,
            'detections': [],
            'summary': {
                'total_cells': 0,
                'counts': {},
                'sickle_ratio': 0.0,
                'diagnosis': 'No valid crops'
            }
        }

    # Batch classify all crops at once
    batch = torch.stack(crop_tensors).to(DEVICE)

    with torch.no_grad():
        outputs = classifier(batch)
        probs = torch.softmax(outputs, dim=1)
        pred_indices = probs.argmax(dim=1)
        pred_confs = probs.max(dim=1).values

    # Merge YOLO boxes with R-CNN classifications
    for i, bd in enumerate(box_data):
        cls_idx = pred_indices[i].item()
        cls_conf = pred_confs[i].item()
        cls_name = class_names[cls_idx] if cls_idx < len(class_names) else f'class_{cls_idx}'

        detections.append({
            'box': bd['box'],
            'yolo_conf': bd['yolo_conf'],
            'class': cls_name,
            'class_idx': cls_idx,
            'class_conf': cls_conf,
            'all_probs': probs[i].cpu().numpy(),
        })

    # --- Stage 3: Aggregate Results ---
    total = len(detections)
    counts = {}
    for d in detections:
        c = d['class']
        counts[c] = counts.get(c, 0) + 1

    # Calculate sickle ratio (check for various possible class names)
    sickle_keywords = ['sickle', 'positive', 'abnormal', 'scd']
    sickle_count = 0
    for cls_name, cnt in counts.items():
        if any(kw in cls_name.lower() for kw in sickle_keywords):
            sickle_count += cnt

    sickle_ratio = sickle_count / total if total > 0 else 0.0

    if sickle_ratio > SICKLE_RATIO_THRESHOLD:
        diagnosis = 'SCD Positive'
    else:
        diagnosis = 'SCD Negative'

    summary = {
        'total_cells': total,
        'counts': counts,
        'sickle_count': sickle_count,
        'sickle_ratio': sickle_ratio,
        'diagnosis': diagnosis,
        'avg_class_conf': np.mean([d['class_conf'] for d in detections]),
    }

    return {
        'image': img,
        'detections': detections,
        'summary': summary
    }


print('Hybrid pipeline defined.')

In [ ]:
# ============================================================
# CELL 5: Visualization Function
# ============================================================

def visualize_result(result, title='Hybrid Pipeline Result'):
    """
    Draw bounding boxes colored by R-CNN classification on the image.
    Green = Normal, Red = Sickle/Positive, Cyan = other classes.
    """
    img = result['image']
    detections = result['detections']
    summary = result['summary']

    fig, ax = plt.subplots(1, 1, figsize=(14, 14))
    ax.imshow(img)

    # Color map: sickle-related = red, normal-related = green, other = cyan
    sickle_keywords = ['sickle', 'positive', 'abnormal', 'scd']
    normal_keywords = ['normal', 'negative', 'healthy']

    for det in detections:
        x1, y1, x2, y2 = det['box']
        cls_name = det['class']
        cls_conf = det['class_conf']
        yolo_conf = det['yolo_conf']

        # Determine color
        cls_lower = cls_name.lower()
        if any(kw in cls_lower for kw in sickle_keywords):
            color = 'red'
        elif any(kw in cls_lower for kw in normal_keywords):
            color = 'lime'
        else:
            color = 'cyan'

        # Draw box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)

        # Label: class name + confidence
        label = f'{cls_name} {cls_conf:.0%}'
        ax.text(
            x1, y1 - 5, label, color='white', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8)
        )

    # Title with summary
    counts_str = ', '.join([f'{k}: {v}' for k, v in summary['counts'].items()])
    diag = summary['diagnosis']
    ratio = summary['sickle_ratio']
    total = summary['total_cells']

    ax.set_title(
        f'{title}\n'
        f'Cells: {total} | {counts_str}\n'
        f'Sickle Ratio: {ratio:.1%} | Diagnosis: {diag}',
        fontsize=13, fontweight='bold',
        color='red' if 'Positive' in diag else 'green'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()


print('Visualization function defined.')

In [ ]:
# ============================================================
# CELL 6: Run Pipeline on Test Images
# ============================================================

# Collect test images
if os.path.isdir(TEST_IMAGES_DIR):
    test_images = sorted([
        os.path.join(TEST_IMAGES_DIR, f)
        for f in os.listdir(TEST_IMAGES_DIR)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
else:
    test_images = [TEST_IMAGES_DIR]  # Single file

print(f'Found {len(test_images)} test images')

# Run pipeline on each
all_results = []

for img_path in test_images:
    print(f'\nProcessing: {os.path.basename(img_path)}')

    result = run_hybrid_pipeline(
        img_path, detector, classifier, RCNN_CLASS_NAMES,
        conf_thresh=YOLO_CONF_THRESH,
        iou_thresh=YOLO_IOU_THRESH,
        padding=CROP_PADDING
    )

    all_results.append(result)

    s = result['summary']
    print(f'  Detected: {s["total_cells"]} cells')
    print(f'  Counts: {s["counts"]}')
    print(f'  Sickle ratio: {s["sickle_ratio"]:.1%}')
    print(f'  Diagnosis: {s["diagnosis"]}')
    if s['total_cells'] > 0:
        print(f'  Avg classification confidence: {s["avg_class_conf"]:.1%}')

    visualize_result(result, title=os.path.basename(img_path))

In [ ]:
# ============================================================
# CELL 7: Aggregate Summary Across All Images
# ============================================================

print('=' * 60)
print('HYBRID PIPELINE — AGGREGATE RESULTS')
print('=' * 60)

total_cells_all = 0
total_sickle_all = 0
class_totals = {}
all_confs = []

for i, result in enumerate(all_results):
    s = result['summary']
    total_cells_all += s['total_cells']
    total_sickle_all += s.get('sickle_count', 0)

    for cls, cnt in s['counts'].items():
        class_totals[cls] = class_totals.get(cls, 0) + cnt

    for det in result['detections']:
        all_confs.append(det['class_conf'])

print(f'\nImages processed: {len(all_results)}')
print(f'Total cells detected: {total_cells_all}')
print(f'\nPer-class counts:')
for cls, cnt in sorted(class_totals.items()):
    pct = cnt / total_cells_all * 100 if total_cells_all > 0 else 0
    print(f'  {cls}: {cnt} ({pct:.1f}%)')

overall_ratio = total_sickle_all / total_cells_all if total_cells_all > 0 else 0
print(f'\nOverall sickle ratio: {overall_ratio:.1%}')
print(f'Overall diagnosis: {"SCD Positive" if overall_ratio > SICKLE_RATIO_THRESHOLD else "SCD Negative"}')
if all_confs:
    print(f'Avg classification confidence: {np.mean(all_confs):.1%}')
    print(f'Min classification confidence: {np.min(all_confs):.1%}')

In [ ]:
# ============================================================
# CELL 8: Per-Cell Detail View
# ============================================================
# Show individual cropped cells with their classifications

def show_cell_crops(result, max_cells=20):
    """Show individual cell crops with R-CNN predictions."""
    img = result['image']
    detections = result['detections'][:max_cells]

    if not detections:
        print('No detections to show.')
        return

    n = len(detections)
    cols = min(6, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).flatten()

    sickle_keywords = ['sickle', 'positive', 'abnormal', 'scd']

    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['box']
        # Add padding for display
        w, h = img.size
        bw = x2 - x1
        bh = y2 - y1
        pad = 0.15
        cx1 = max(0, int(x1 - bw * pad))
        cy1 = max(0, int(y1 - bh * pad))
        cx2 = min(w, int(x2 + bw * pad))
        cy2 = min(h, int(y2 + bh * pad))

        crop = img.crop((cx1, cy1, cx2, cy2))
        axes[i].imshow(crop)

        cls_name = det['class']
        conf = det['class_conf']
        is_sickle = any(kw in cls_name.lower() for kw in sickle_keywords)
        color = 'red' if is_sickle else 'green'

        axes[i].set_title(f'{cls_name}\n{conf:.1%}', color=color, fontsize=9, fontweight='bold')
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Individual Cell Classifications', fontsize=13)
    plt.tight_layout()
    plt.show()


# Show crops from the first test image
if all_results:
    show_cell_crops(all_results[0], max_cells=24)

In [ ]:
# ============================================================
# CELL 9: Reusable Inference Function (for Django backend)
# ============================================================
# This cell shows the exact logic your Django backend should use.

def predict_blood_smear(image_path, detector, classifier, class_names,
                         conf=0.25, sickle_threshold=0.3):
    """
    Full hybrid inference on a single blood smear image.

    Returns:
        dict with 'diagnosis', 'sickle_ratio', 'total_cells',
             'sickle_count', 'normal_count', 'confidence',
             'cell_details' (list of per-cell results)
    """
    result = run_hybrid_pipeline(
        image_path, detector, classifier, class_names,
        conf_thresh=conf, padding=0.1
    )

    s = result['summary']

    return {
        'diagnosis': s['diagnosis'],
        'sickle_ratio': round(s['sickle_ratio'], 4),
        'total_cells': s['total_cells'],
        'sickle_count': s.get('sickle_count', 0),
        'normal_count': s['total_cells'] - s.get('sickle_count', 0),
        'confidence': round(s.get('avg_class_conf', 0), 4),
        'counts': s['counts'],
        'cell_details': [
            {
                'box': det['box'],
                'class': det['class'],
                'confidence': round(det['class_conf'], 4)
            }
            for det in result['detections']
        ]
    }


# Demo
if test_images:
    output = predict_blood_smear(
        test_images[0], detector, classifier, RCNN_CLASS_NAMES
    )
    print('=== Backend Output Format ===')
    for k, v in output.items():
        if k == 'cell_details':
            print(f'  cell_details: [{len(v)} cells]')
            for cell in v[:3]:
                print(f'    {cell}')
            if len(v) > 3:
                print(f'    ... ({len(v) - 3} more)')
        else:
            print(f'  {k}: {v}')